# Part 2: Agent Implementation

In [1]:
try:
    __import__('pysqlite3')
    import sys
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
except ImportError:
    pass

import os
import json
import chromadb
import chromadb.utils.embedding_functions as embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from tavily import TavilyClient

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

## Reconnect to ChromaDB

In [2]:
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL
)
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

## Tools

In [3]:
@tool
def retrieve_game(query: str) -> str:
    """Semantic search: Finds most relevant results in the vector DB.
    
    args:
    - query: a question about game industry. 
    
    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3)
    output = []
    for doc, meta, distance in zip(
        results["documents"][0], 
        results["metadatas"][0], 
        results["distances"][0]
    ):
        output.append(f"[Distance: {distance:.4f}] {doc}\n  Metadata: {json.dumps(meta)}")
    return "\n\n".join(output)

In [4]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    eval_llm = LLM(
        model="gpt-4o-mini", 
        temperature=0.0
    )
    
    class EvaluationReport(BaseModel):
        useful: bool = Field(description="Whether the documents are useful to answer the question")
        description: str = Field(description="Description about the evaluation result")
    
    prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

Question: {question}
Retrieved Documents: {retrieved_docs}

Evaluate whether these documents contain sufficient and relevant information to answer the question."""
    
    response = eval_llm.invoke(prompt, response_format=EvaluationReport)
    return response.content

In [5]:
@tool
def game_web_search(question: str) -> str:
    """Searches the web for information about video games when internal knowledge is insufficient.
    
    args:
    - question: a question about game industry.
    """
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
    response = tavily_client.search(query=question, max_results=3)
    
    results = []
    for result in response.get("results", []):
        results.append(f"Source: {result['url']}\nContent: {result['content']}")
    
    return "\n\n---\n\n".join(results) if results else "No web results found."

## Create the Agent

In [6]:
SYSTEM_INSTRUCTIONS = """You are UdaPlay, an AI research assistant for the video game industry.

Your workflow:
1. When asked about a game, FIRST use retrieve_game to search the internal database
2. Then use evaluate_retrieval to assess if the results are good enough to answer
3. If evaluation shows results are NOT useful, use game_web_search to find the answer online
4. Provide a clear, structured answer with citations to your sources

Always cite whether your answer came from the internal database or web search.
Be specific with dates, platforms, publishers, and other factual details."""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3
)

## Testing Queries

In [7]:
def print_run(run):
    final_state = run.get_final_state()
    if not final_state: return
    for msg in final_state.get("messages", []):
        print(f"\n{'='*40}")
        if isinstance(msg, UserMessage):
            print(f"USER: {msg.content}")
        elif isinstance(msg, AIMessage):
            if msg.content:
                print(f"AI: {msg.content}")
            if msg.tool_calls:
                for call in msg.tool_calls:
                    print(f"\nCalling Tool: {call.function.name}")
                    print(f"Arguments: {call.function.arguments}")
        elif isinstance(msg, ToolMessage):
            print(f"TOOL RESULT: {msg.content}")


In [8]:
print("\n*** QUERY 1 ***")
run1 = agent.invoke("When was Pokémon Gold and Silver released?", session_id="q1")
print_run(run1)
print(f"\nTokens Used: {run1.get_final_state().get('total_tokens', 0)}")


*** QUERY 1 ***
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


USER: When was Pokémon Gold and Silver released?


Calling Tool: retrieve_game
Arguments: {"query":"Pokémon Gold and Silver release date"}

TOOL RESULT: "[Distance: 0.1306] [Game Boy Color] Pok\u00c3\u00a9mon Gold and Silver (1999) - Second-generation Pok\u00c3\u00a9mon games introducing new regions, Pok\u00c3\u00a9mon, and gameplay mechanics.\n  Metadata: {\"Genre\": \"Role-playing\", \"Publisher\": \"Nintendo\", \"Description\": \"Second-generation Pok\\u00c3\\u00a9mon games introducing new regions, Pok\\u00c3\\u00a9mon, and gameplay mechanics.\", \"Name\": \"Pok\\u00c3\\u00a9mon Gold and Silver\", \"Platform\": \"Game Boy Color\", \"YearOfRelease\": \"1999\"}\n\n[Distance: 0.1644] [Game Boy Advance] Pok\u00c3\u00a9mon Ruby and Sapphire (2002) - Third-generation Pok\u00c3\u00a9mon games set in the Hoenn region, featuring new Pok\u00c3\u00a9mon and double battles.\n  Metadata: {\"Genre\": \"Role

In [9]:
print("\n*** QUERY 2 ***")
run2 = agent.invoke("Which one was the first 3D platformer Mario game?", session_id="q2")
print_run(run2)
print(f"\nTokens Used: {run2.get_final_state().get('total_tokens', 0)}")


*** QUERY 2 ***
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


USER: Which one was the first 3D platformer Mario game?


Calling Tool: retrieve_game
Arguments: {"query":"first 3D platformer Mario game"}

TOOL RESULT: "[Distance: 0.1032] [Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.\n  Metadata: {\"Genre\": \"Platformer\", \"Publisher\": \"Nintendo\", \"Name\": \"Super Mario 64\", \"YearOfRelease\": \"1996\", \"Description\": \"A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.\", \"Platform\": \"Nintendo 64\"}\n\n[Distance: 0.1278] [Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.\n  Metadata: {\"Genre\": \"Platformer\", \"Name\": \"Super Mario World\",

In [10]:
print("\n*** QUERY 3 ***")
run3 = agent.invoke("Was Mortal Kombat X released for PlayStation 5?", session_id="q3")
print_run(run3)
print(f"\nTokens Used: {run3.get_final_state().get('total_tokens', 0)}")


*** QUERY 3 ***
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


USER: Was Mortal Kombat X released for PlayStation 5?


Calling Tool: retrieve_game
Arguments: {"query":"Mortal Kombat X PlayStation 5 release"}

TOOL RESULT: "[Distance: 0.1790] [PlayStation 5] Marvel's Spider-Man 2 (2023) - The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.\n  Metadata: {\"Description\": \"The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.\", \"Genre\": \"Action-adventure\", \"YearOfRelease\": \"2023\", \"Publisher\": \"Sony Interactive Entertainment\", \"Name\": \"Marvel's Spider-Man 2\", \"Platform\": \"PlayStation 5\"}\n\n[Distance: 0.1984] [PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.\n  Metadata: {\"Name\": \"Marvel's Spider-Man\", \"